# 📊 Análise Exploratória (EDA) — PI5: Violência Contra a Mulher

**Objetivo:** Extrair insights estatísticos e visuais dos dados carregados no BigQuery (Sorocaba e Votorantim).

---
## 🛠️ Configuração Inicial

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

# Configuração de Estilo
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16

# Autenticação
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../credenciais.json"
client = bigquery.Client()

print("✅ Ambiente configurado e conectado ao BigQuery!")

## 📥 Carregamento dos Dados

Buscando os dados consolidados diretamente da tabela no BigQuery.

In [ ]:
PROJECT_ID = 'projetointegrador5'
TABLE_ID = 'dados_ssp.dados_ddm'

query = f"SELECT * FROM `{PROJECT_ID}.{TABLE_ID}`"
df = client.query(query).to_dataframe()

# --- GARANTE TIPOS DE DADOS PARA O PANDAS ---
df['data_ocorrencia_bo'] = pd.to_datetime(df['data_ocorrencia_bo'])

print(f"✅ Dados carregados e tipados: {df.shape[0]} registros e {df.shape[1]} colunas.")
df.head()

## 📈 Tendências Temporais

Como o número de ocorrências evoluiu ao longo dos meses e anos?

In [ ]:
# Ordenando por data
df_temporal = df.copy()
df_temporal['mes_ano'] = df_temporal['data_ocorrencia_bo'].dt.to_period('M').astype(str)

plt.figure(figsize=(15, 6))
sns.countplot(data=df_temporal.sort_values('data_ocorrencia_bo'), x='mes_ano', hue='nome_municipio')
plt.title('Evolução Mensal de Ocorrências por Município')
plt.xticks(rotation=45)
plt.legend(title='Município')
plt.tight_layout()
plt.show()

## ⚔️ Tipificação Criminal

Quais são os crimes mais comuns registrados nas DDMs?

In [ ]:
plt.figure(figsize=(10, 8))
crime_counts = df['tipo_ocorrencia'].value_counts().head(10)
sns.barplot(x=crime_counts.values, y=crime_counts.index, palette='viridis')
plt.title('Top 10 Tipos de Ocorrências mais Frequentes')
plt.xlabel('Quantidade de Registros')
plt.show()

## 🕒 Período e Dia da Semana

Existe algum padrão de horários ou dias com mais registros?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

# Dia da Semana
ordem_dias = ['Segunda-Feira', 'Terça-Feira', 'Quarta-Feira', 'Quinta-Feira', 'Sexta-Feira', 'Sábado', 'Domingo']
sns.countplot(data=df, x='dia_semana', order=ordem_dias, ax=ax[0], palette='magma')
ax[0].set_title('Ocorrências por Dia da Semana')
ax[0].tick_params(axis='x', rotation=45)

# Período do Dia
sns.countplot(data=df, x='periodo_ocorrencia', ax=ax[1], palette='flare')
ax[1].set_title('Ocorrências por Período do Dia')

plt.tight_layout()
plt.show()

## 📍 Análise Geográfica (Bairros)

Quais bairros apresentam maior concentração de registros em Sorocaba?

In [ ]:
df_sorocaba = df[df['nome_municipio'] == 'Sorocaba']
top_bairros = df_sorocaba['bairro'].value_counts().head(15)

plt.figure(figsize=(10, 8))
sns.barplot(x=top_bairros.values, y=top_bairros.index, palette='rocket')
plt.title('Top 15 Bairros com mais Ocorrências (Sorocaba)')
plt.xlabel('Quantidade')
plt.show()

## 📝 Conclusões Preliminares

1. **Tipo de Crime:** A Lesão Corporal Dolosa e Estupro de Vulnerável costumam ser predominantes nas DDMs.
2. **Padrão Temporal:** As ocorrências são distribuídas ao longo da semana, com destaque para domingo e segunda-feira.
3. **Foco Geográfico:** Identificamos os bairros que exigem maior atenção das políticas públicas de proteção à mulher.

---